# MEC_4OW05_TA: Numerical Simulation Methods
## Lab 01: Initial Value Problems & Hydrodynamic Buoy Dynamics

---

### Objectives
1. Understand 1st-order vector time-stepping and energy conservation.
2. Implement **Heun's 2nd-Order Runge-Kutta method (RK2)**.
3. Formulate the state-space equations for a **floating buoy** subject to wave forcing and non-linear drag ($v\vert{}v\vert{}$).
4. Identify the **numerical stability boundary** of explicit schemes and observe how physical non-linear drag limits resonant motions.

### Deliverables
* **In-Class (Oral Check-Off ~17:00):** Demonstrate your working buoy response plots to the instructor.
* **Final Submission:** Answer the short analytical questions at the end of this notebook and hand in your completed `.ipynb` file.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["font.size"] = 11
plt.rcParams["lines.linewidth"] = 1.8

---
## Section 1: Pre-Implemented Reference Solvers

Standard **Forward Euler** and **Classical RK4** are pre-implemented below for general 1st-order systems $\frac{d\mathbf{y}}{dt} = \mathbf{f}(t, \mathbf{y})$:

In [ ]:
def solve_euler(rhs_func, y0, t_grid, *args):
    """Forward Euler integrator for vector ODE systems."""
    y0 = np.asarray(y0, dtype=float)
    N_steps = len(t_grid)
    y_sol = np.zeros((N_steps, len(y0)))
    y_sol[0] = y0
    for n in range(N_steps - 1):
        dt = t_grid[n+1] - t_grid[n]
        y_sol[n+1] = y_sol[n] + dt * rhs_func(t_grid[n], y_sol[n], *args)
    return y_sol


def solve_rk4(rhs_func, y0, t_grid, *args):
    """Classical 4th-Order Runge-Kutta integrator."""
    y0 = np.asarray(y0, dtype=float)
    N_steps = len(t_grid)
    y_sol = np.zeros((N_steps, len(y0)))
    y_sol[0] = y0
    for n in range(N_steps - 1):
        dt = t_grid[n+1] - t_grid[n]
        tn, yn = t_grid[n], y_sol[n]
        k1 = rhs_func(tn, yn, *args)
        k2 = rhs_func(tn + 0.5*dt, yn + 0.5*dt*k1, *args)
        k3 = rhs_func(tn + 0.5*dt, yn + 0.5*dt*k2, *args)
        k4 = rhs_func(tn + dt, yn + dt*k3, *args)
        y_sol[n+1] = yn + (dt / 6.0) * (k1 + 2.0*k2 + 2.0*k3 + k4)
    return y_sol

---
## Section 2 (Task 1): Implement Heun's Method (RK2)

### Mathematical Formulation
Heun's 2nd-order Runge-Kutta method predicts the state at $t_{n+1}$ using Euler, evaluates the slope at the predicted point, and averages the slopes:

$$\begin{aligned} \mathbf{k}_1 &= \mathbf{f}(t_n, \; \mathbf{y}_n) \\ \mathbf{k}_2 &= \mathbf{f}(t_n + \Delta t, \; \mathbf{y}_n + \Delta t \mathbf{k}_1) \\ \mathbf{y}_{n+1} &= \mathbf{y}_n + \frac{\Delta t}{2} \left( \mathbf{k}_1 + \mathbf{k}_2 \right) \end{aligned}$$

### What is expected:
Fill in the calculation for `k1`, `k2`, and `y_sol[n+1]` in `solve_heun` below.

In [ ]:
def solve_heun(rhs_func, y0, t_grid, *args):
    """Heun 2nd-Order Runge-Kutta (RK2) Integrator."""
    y0 = np.asarray(y0, dtype=float)
    N_steps = len(t_grid)
    y_sol = np.zeros((N_steps, len(y0)))
    y_sol[0] = y0
    
    for n in range(N_steps - 1):
        dt = t_grid[n+1] - t_grid[n]
        tn = t_grid[n]
        yn = y_sol[n]
        
        # ==========================================================
        # TODO: Compute slopes k1, k2 and update y_sol[n+1]
        # ==========================================================
        pass
        
    return y_sol

---
## Section 3: Diagnostic Benchmark — Energy Conservation

Consider an undamped physical oscillator with mass $m = 1000\text{ kg}$ and stiffness $K = 4000\text{ N/m}$:

$$\frac{d}{dt}\begin{bmatrix} x \\ v \end{bmatrix} = \begin{bmatrix} v \\ -\frac{K}{m}x \end{bmatrix}$$

The total mechanical energy must remain constant:
$$E(t) = \frac{1}{2}m v(t)^2 + \frac{1}{2}K x(t)^2 = E(0)$$

Run the cell below to inspect how Forward Euler, Heun, and RK4 behave regarding numerical energy gain/loss:

In [ ]:
def linear_oscillator_rhs(t, y, m=1000.0, K=4000.0):
    return np.array([y[1], -(K / m) * y[0]])

t_grid = np.arange(0, 20.1, 0.1)
y0 = [1.0, 0.0]  # Initial displacement 1 m, at rest

sol_euler = solve_euler(linear_oscillator_rhs, y0, t_grid)
sol_heun  = solve_heun(linear_oscillator_rhs, y0, t_grid)
sol_rk4   = solve_rk4(linear_oscillator_rhs, y0, t_grid)

E0 = 0.5 * 4000.0 * 1.0**2

def get_energy(sol):
    return (0.5 * 1000.0 * sol[:, 1]**2 + 0.5 * 4000.0 * sol[:, 0]**2) / E0

plt.figure()
plt.plot(t_grid, get_energy(sol_euler), "r--", label="Forward Euler (Artificial Gain)")
plt.plot(t_grid, get_energy(sol_heun),  "g-.", label="Heun RK2")
plt.plot(t_grid, get_energy(sol_rk4),   "b-",  label="Classical RK4 (Conserved)")
plt.axhline(1.0, color="k", ls=":", alpha=0.6, label="Exact E(t)/E(0) = 1")
plt.xlabel("Time t [s]")
plt.ylabel("Normalized Energy E(t) / E(0)")
plt.title("Energy Drift on Undamped Oscillator (dt = 0.1 s)")
plt.ylim(0.5, 2.5)
plt.legend()
plt.grid(True, ls=":")
plt.show()

---
## Section 4 (Task 2): Hydrodynamic Floating Buoy RHS

### Physical Problem
A floating cylindrical buoy undergoes 1-DOF vertical heave motion $x(t)$ excited by waves, subject to restoring hydrostatic stiffness, linear damping, and non-linear quadratic drag:

$$(m + a_\infty) \ddot{x}(t) + d_{\text{lin}} \dot{x}(t) + d_{\text{quad}} \dot{x}(t)\vert{}\dot{x}(t)\vert{} + K x(t) = F_0 \cos(\omega t)$$

| Parameter | Meaning | Value |
| :--- | :--- | :--- |
| $m$ | Buoy dry mass | $1000\text{ kg}$ |
| $a_\infty$ | Heave added mass | $250\text{ kg}$ |
| $d_{\text{lin}}$ | Linear radiation damping | $80\text{ N}\cdot\text{s/m}$ |
| $d_{\text{quad}}$ | Non-linear quadratic drag | $150\text{ N}\cdot\text{s}^2/\text{m}^2$ |
| $K$ | Hydrostatic stiffness | $5000\text{ N/m}$ |
| $F_0$ | Wave force amplitude | $1200\text{ N}$ |
| $\omega$ | Wave frequency | $1.8\text{ rad/s}$ |

### What is expected:
Reduce this 2nd-order ODE to 1st-order form $\mathbf{y} = [x, v]^T$ and complete `spar_buoy_rhs` below:

In [ ]:
def spar_buoy_rhs(t, y, m=1000.0, a_inf=250.0, d_lin=80.0, d_quad=150.0, K=5000.0, F0=1200.0, omega=1.8):
    """
    RHS for floating buoy heave motion.
    y[0] = heave displacement x [m]
    y[1] = heave velocity v [m/s]
    """
    x = y[0]
    v = y[1]
    
    F_wave = F0 * np.cos(omega * t)
    total_mass = m + a_inf
    
    # ==========================================================
    # TODO: Express dxdt and dvdt
    # dvdt = (F_wave - linear_drag - quadratic_drag - restoring) / total_mass
    # Hint: use np.abs(v) for non-linear drag v*|v|
    # ==========================================================
    dxdt = v
    dvdt = 0.0
    
    return np.array([dxdt, dvdt])

---
## Section 5 (Task 3): Simulate Buoy Response & Oral Check-Off

### What is expected:
Simulate the buoy motion from rest $\mathbf{y}(0) = [0, 0]^T$ over $40\text{ s}$ with $\Delta t = 0.05\text{ s}$ using `solve_rk4`.  
Plot both displacement $x(t)$ and velocity $v(t)$ in stacked subplots.

In [ ]:
t_grid = np.arange(0, 40.05, 0.05)
y0 = [0.0, 0.0]

# ==========================================================
# TODO:
# 1. Compute solution using solve_rk4
# 2. Plot displacement x(t) and velocity v(t) in 2 subplots
# ==========================================================


---
## Section 6: Analysis Questions

Type your answers in this cell:

### Question 1: Resonant Amplification & Drag
* The natural frequency is $\omega_n = \sqrt{K / (m + a_\infty)} = \sqrt{5000 / 1250} = 2.0\text{ rad/s}$.
* Set `omega = 2.0` in `spar_buoy_rhs` and re-run.
* Now set `d_quad = 0.0` (linear damping only). What happens to the maximum heave amplitude? Why is non-linear quadratic drag vital for marine structures in severe sea states?

**Your Answer:**  
*(Write 2-3 sentences here)*

---

### Question 2: Forward Euler Stability Limit
* Re-run the buoy simulation using `solve_euler` instead of `solve_rk4`.
* Try time steps $\Delta t = 0.02\text{ s}, 0.05\text{ s}, 0.1\text{ s}, 0.2\text{ s}$.
* At approximately what step size does Forward Euler blow up? How does this connect to the eigenvalue stability condition on the complex plane from lecture?

**Your Answer:**  
*(Write 2-3 sentences here)*